# Portfolio Metric Forecasting — Advanced Re-Forecasting
## Fingerhut | Beginning Outstanding Loans
---
**Benchmarks to beat:**
- `Original` forecast MAPE: 395.19%
- `2025 0+12` re-forecast MAPE: 3.21% ← **current statistical target**
- `Senior Prophet` baseline MAPE: 6.54%

**Strategy:**
1. Tune Prophet hyperparameters via Bayesian optimisation (Hyperopt / TPE)
2. Benchmark classical statistical models (MA → SARIMA)
3. Benchmark regularised ML models (Ridge → XGBoost / LightGBM)

> **Data note:** ~19 monthly training points (2023-05 → 2024-12).  All model and feature choices are deliberately conservative to avoid over-fitting.


## 1. Installations

In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn
!pip install --proxy=192.168.2.10:8080 prophet
!pip install --proxy=192.168.2.10:8080 hyperopt          # Bayesian optimisation
!pip install --proxy=192.168.2.10:8080 pmdarima           # Auto-ARIMA
!pip install --proxy=192.168.2.10:8080 xgboost
!pip install --proxy=192.168.2.10:8080 lightgbm


## 2. Imports

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector
import datetime

# Forecasting / Statistics
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

# ML
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from xgboost import XGBRegressor
import lightgbm as lgb

# Hyperopt
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.float_format', '{:,.2f}'.format)


## 3. Setting Up the Connector

In [ ]:
# load the credentials
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']
ctx = snowflake.connector.connect(**snowflake_creds)

In [ ]:
cur = ctx.cursor()

## 4. Load Datasets

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
df.head()

## 5. Data Preparation

In [ ]:
portfolio      = 'Fingerhut'
last_input_date = '2024-12-31'

portfolio_df = df[df['PORTFOLIO'] == portfolio]
portfolio_df = portfolio_df[portfolio_df['FORECAST_TYPE'] == 'Actual']
metrics = portfolio_df['METRIC'].unique()
metrics

In [ ]:
dff = portfolio_df.groupby(['DATE', 'METRIC']).sum().reset_index()
dff

In [ ]:
data_df = dff.pivot(
    index='DATE',
    columns='METRIC',
    values='METRIC_VALUE'
).sort_index()

# Split into train / test
train_df = data_df[data_df.index <= datetime.date.fromisoformat(last_input_date)]
test_df  = data_df[data_df.index >  datetime.date.fromisoformat(last_input_date)]

print(f'Training samples : {len(train_df)}')
print(f'Test samples     : {len(test_df)}')
print(f'Training period  : {train_df.index.min()} → {train_df.index.max()}')
train_df.head()

## 6. Helper Functions (from existing notebook)

In [ ]:
# ── Plotting helper ──────────────────────────────────────────────────────
def plot_results(pred_df, metric):
    plt.figure(figsize=(20, 6))
    plt.title(metric)
    sns.lineplot(data=pred_df, x='DATE', y='METRIC_VALUE',
                 hue='FORECAST_TYPE', errorbar=None)
    plt.xlabel('DATE')
    plt.ylabel('METRIC_VALUE')
    plt.xticks(pred_df['DATE'].unique(), rotation=90)
    plt.tight_layout()
    plt.show()


# ── Original Prophet predict (senior baseline — do NOT modify) ───────────
def predict_metrics(train_df, metric):
    pred_df_train = train_df[metric].reset_index().rename(
        columns={'DATE': 'ds', metric: 'y'})

    model = Prophet(
        growth='logistic',
        changepoint_prior_scale=1.0,
        seasonality_mode='additive',
        seasonality_prior_scale=1,
        weekly_seasonality=False,
        daily_seasonality=False,
        yearly_seasonality=False,
        changepoint_range=0.95,
        n_changepoints=60,
    )

    pred_df_train['floor'] = 0
    pred_df_train['cap']   = 1.2 * pred_df_train['y'].max()
    test_df['floor']       = 0
    test_df['cap']         = 1.2 * pred_df_train['y'].max()

    print(pred_df_train.columns)
    model.fit(pred_df_train)

    pred_df = model.predict(
        test_df.reset_index(names='ds'))[['ds', 'yhat']] \
        .rename(columns={'ds': 'DATE', 'yhat': 'PRED'}) \
        .set_index('DATE')

    pred_df = pred_df.merge(test_df.loc[pred_df.index, metric],
                            left_index=True, right_index=True, how='inner')\
        .rename(columns={metric: 'Actual'})

    pred_df = pred_df.reset_index().melt(
        id_vars='DATE',
        value_vars=['Actual', 'PRED'],
    ).rename(columns={'variable': 'FORECAST_TYPE', 'value': 'METRIC_VALUE'})

    return pred_df


def add_reforecast_data(pred_df, main_df, portfolio, metric, forecast_types):
    df = main_df[
        (main_df['PORTFOLIO'] == portfolio) &
        (main_df['DATE'].astype(str).isin(pred_df['DATE'].astype(str))) &
        (main_df['METRIC'] == metric) &
        (main_df['FORECAST_TYPE'].isin(forecast_types))   
    ]
    df = df[pred_df.columns]
    df['DATE'] = df['DATE'].apply(
        lambda x: datetime.date.fromisoformat(str(x)))
    pred_df = pd.concat([pred_df, df], axis=0)
    return pred_df


# ── Evaluate helper ──────────────────────────────────────────────────────
def evaluate(df_check, start, end, metric,
             reforecast_cols=['2025 9+3'],
             portfolio='PFCP',
             original_col='Original'):
    df_check  = df_check[df_check['METRIC'] == metric]
    df_check  = df_check[df_check['FORECAST_TYPE'].isin(
        [original_col, 'Actual'] + reforecast_cols)]

    start_date = datetime.date.fromisoformat(start)
    end_date   = datetime.date.fromisoformat(end)
    df_check   = df_check[(df_check['DATE'] >= start_date) &
                          (df_check['DATE'] <= end_date)]

    y_true = df_check[df_check['FORECAST_TYPE'] == 'Actual'] \
        .sort_values('DATE').set_index('DATE')['METRIC_VALUE']

    result_df = pd.DataFrame(index=y_true.index, columns=['Actual'])
    result_df['Actual'] = y_true.values

    for col in [original_col] + reforecast_cols:
        y_pred          = df_check[df_check['FORECAST_TYPE'] == col] \
            .sort_values('DATE').set_index('DATE')['METRIC_VALUE']
        y_pred.name     = col
        result_df       = result_df.merge(
            y_pred, how='left', left_index=True, right_index=True)

    return result_df


## 7. Extract Target Metric Series

In [ ]:
# Focus metric for this notebook
target_metric = 'Beginning Outstanding Loans'

train_series = train_df[target_metric].dropna()
test_series  = test_df[target_metric].dropna()  if len(test_df) > 0 else pd.Series(dtype=float)

print(f'Target  : {target_metric}')
print(f'Train n : {len(train_series)} | {train_series.index.min()} → {train_series.index.max()}')
print(f'Test  n : {len(test_series)}')

# Number of future periods to forecast (12-month horizon)
N_FORECAST = 12

# Future dates for forecast output
last_train_date = pd.Timestamp(train_series.index.max())
forecast_dates  = pd.date_range(
    start=last_train_date + pd.DateOffset(months=1),
    periods=N_FORECAST, freq='MS'
).date
print(f'Forecast dates: {forecast_dates[0]} → {forecast_dates[-1]}')

# Quick visualisation of training data
plt.figure(figsize=(14, 4))
plt.plot(train_series.index, train_series.values, marker='o', label='Actual (train)')
if len(test_series) > 0:
    plt.plot(test_series.index, test_series.values,
             marker='s', linestyle='--', color='orange', label='Actual (test)')
plt.title(f'{target_metric} — {portfolio}')
plt.xlabel('DATE'); plt.ylabel('VALUE')
plt.xticks(rotation=45); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Dictionary to collect all model MAPEs for final comparison
results_mape = {}   # {model_name: {'cv_mape': float, 'test_mape': float or None}}

# ── Utility: compute MAPE safely (skip NaN, skip zero actuals) ───────────
def safe_mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = (~np.isnan(y_true)) & (~np.isnan(y_pred)) & (y_true != 0)
    if mask.sum() == 0:
        return np.nan
    return mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100

# ── Utility: time-series walk-forward splits (no data leakage) ───────────
def ts_cv_splits(series, n_splits=3, val_size=2, min_train=10):
    """Yield (train_idx, val_idx) index pairs for walk-forward CV."""
    n = len(series)
    for fold in range(n_splits):
        val_end   = n - (n_splits - 1 - fold) * val_size
        val_start = val_end - val_size
        if val_start < min_train:
            continue
        yield list(range(val_start)), list(range(val_start, val_end))


## 8. Prophet — Bayesian Hyperparameter Tuning (Hyperopt / TPE)

### Search space
| Parameter | Range | Explanation |
|---|---|---|
| `changepoint_prior_scale` | log-uniform [0.001, 10] | trend flexibility |
| `seasonality_prior_scale` | log-uniform [0.01, 5] | seasonality strength |
| `changepoint_range` | uniform [0.70, 0.95] | history fraction for changepoints |
| `n_changepoints` | int [5, 30] | maximum changepoints (keep low — small data) |

> **CV strategy:** 3-fold walk-forward, `val_size=2` months per fold.  
> Loss = mean MAPE across folds.


In [ ]:
# ── Hyperopt search space ────────────────────────────────────────────────
prophet_space = {
    'changepoint_prior_scale': hp.loguniform(
        'changepoint_prior_scale', np.log(0.001), np.log(10)),
    'seasonality_prior_scale': hp.loguniform(
        'seasonality_prior_scale', np.log(0.01), np.log(5)),
    'changepoint_range': hp.uniform('changepoint_range', 0.70, 0.95),
    'n_changepoints'  : hp.quniform('n_changepoints', 5, 30, 1),
}


# ── Cross-validation objective ────────────────────────────────────────────
def prophet_cv_objective(params):
    """
    Walk-forward CV MAPE for Prophet with the given hyperparameters.
    Returns a dict compatible with Hyperopt.
    """
    fold_mapes = []

    for tr_idx, val_idx in ts_cv_splits(train_series, n_splits=3,
                                         val_size=2, min_train=10):
        tr  = train_series.iloc[tr_idx]
        val = train_series.iloc[val_idx]

        # Build Prophet training frame
        df_p = pd.DataFrame({
            'ds': pd.to_datetime(tr.index),
            'y' : tr.values
        })
        df_p['floor'] = 0
        df_p['cap']   = df_p['y'].max() * 1.2

        # Build future frame (validation period)
        df_val = pd.DataFrame({
            'ds'   : pd.to_datetime(val.index),
            'floor': 0,
            'cap'  : df_p['y'].max() * 1.2
        })

        try:
            m = Prophet(
                growth='logistic',
                changepoint_prior_scale = params['changepoint_prior_scale'],
                seasonality_prior_scale = params['seasonality_prior_scale'],
                changepoint_range       = params['changepoint_range'],
                n_changepoints          = int(params['n_changepoints']),
                seasonality_mode        = 'additive',
                weekly_seasonality      = False,
                daily_seasonality       = False,
                yearly_seasonality      = False,
            )
            m.fit(df_p)
            fc   = m.predict(df_val)['yhat'].values
            fc   = np.maximum(fc, 0)   # floor at 0
            mape = safe_mape(val.values, fc)
            if not np.isnan(mape):
                fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)   # heavy penalty for crashes

    loss = np.mean(fold_mapes) if fold_mapes else 200
    return {'loss': loss, 'status': STATUS_OK}


In [ ]:
# ── Run Hyperopt (50 trials) ──────────────────────────────────────────────
print('Running Hyperopt optimisation for Prophet — please wait...')
prophet_trials = Trials()

best_prophet_params_raw = fmin(
    fn         = prophet_cv_objective,
    space      = prophet_space,
    algo       = tpe.suggest,
    max_evals  = 50,
    trials     = prophet_trials,
    rstate     = np.random.default_rng(42),
    verbose    = False,
)

best_prophet_params = space_eval(prophet_space, best_prophet_params_raw)
best_prophet_params['n_changepoints'] = int(best_prophet_params['n_changepoints'])

print('\n=== Best Prophet Hyperparameters ===')
for k, v in best_prophet_params.items():
    print(f'  {k:<30}: {v}')
print(f"\n  Best CV MAPE : {min(prophet_trials.losses()):.2f}%")


In [ ]:
# ── Loss curve across trials ──────────────────────────────────────────────
trial_losses = [t['result']['loss'] for t in prophet_trials.trials]
best_so_far  = [min(trial_losses[:i+1]) for i in range(len(trial_losses))]

plt.figure(figsize=(12, 4))
plt.plot(trial_losses,  alpha=0.4, label='Trial MAPE')
plt.plot(best_so_far,   color='red', linewidth=2, label='Best so far')
plt.axhline(3.21, color='green', linestyle='--', label='2025 0+12 target (3.21%)')
plt.axhline(6.54, color='orange', linestyle='--', label='Senior baseline (6.54%)')
plt.xlabel('Trial'); plt.ylabel('CV MAPE (%)')
plt.title('Hyperopt — Prophet Optimisation')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Train final Prophet with best params (full training data) ────────────
def predict_metrics_tuned(train_df, metric, best_params, forecast_dates):
    """
    Train Prophet with tuned hyperparameters and return prediction dataframe
    in the same format as predict_metrics().
    """
    series = train_df[metric].dropna()

    df_p = pd.DataFrame({
        'ds': pd.to_datetime(series.index),
        'y' : series.values
    })
    cap = df_p['y'].max() * 1.2
    df_p['floor'] = 0
    df_p['cap']   = cap

    m = Prophet(
        growth='logistic',
        changepoint_prior_scale = best_params['changepoint_prior_scale'],
        seasonality_prior_scale = best_params['seasonality_prior_scale'],
        changepoint_range       = best_params['changepoint_range'],
        n_changepoints          = int(best_params['n_changepoints']),
        seasonality_mode        = 'additive',
        weekly_seasonality      = False,
        daily_seasonality       = False,
        yearly_seasonality      = False,
    )
    m.fit(df_p)

    future_df = pd.DataFrame({'ds': pd.to_datetime(forecast_dates)})
    future_df['floor'] = 0
    future_df['cap']   = cap

    fc = m.predict(future_df)[['ds', 'yhat']].rename(
        columns={'ds': 'DATE', 'yhat': 'METRIC_VALUE'})
    fc['DATE']          = fc['DATE'].dt.date
    fc['METRIC_VALUE']  = np.maximum(fc['METRIC_VALUE'], 0)
    fc['FORECAST_TYPE'] = 'Prophet_Tuned'
    return fc


# Run
prophet_tuned_pred = predict_metrics_tuned(
    train_df, target_metric, best_prophet_params, forecast_dates)

# CV MAPE (already computed by hyperopt)
prophet_tuned_cv_mape = min(prophet_trials.losses())

# Test MAPE if test data available
if len(test_series) > 0:
    overlap = [
        d for d in forecast_dates
        if d in test_series.index
    ]
    if overlap:
        pred_overlap = prophet_tuned_pred[
            prophet_tuned_pred['DATE'].isin(overlap)]['METRIC_VALUE'].values
        true_overlap = test_series.loc[overlap].values
        test_mape    = safe_mape(true_overlap, pred_overlap)
        print(f'Prophet Tuned — Test MAPE : {test_mape:.2f}%')
        results_mape['Prophet_Tuned'] = {
            'cv_mape': prophet_tuned_cv_mape, 'test_mape': test_mape}
    else:
        print('No overlap between forecast dates and test_series for MAPE.')
        results_mape['Prophet_Tuned'] = {
            'cv_mape': prophet_tuned_cv_mape, 'test_mape': None}
else:
    results_mape['Prophet_Tuned'] = {
        'cv_mape': prophet_tuned_cv_mape, 'test_mape': None}

print(f'Prophet Tuned — CV MAPE   : {prophet_tuned_cv_mape:.2f}%')
print(f'Senior baseline CV MAPE   : 6.54%')
print(f'Statistical target MAPE   : 3.21%')


## 9. Statistical Models

Models tested (order of increasing complexity):
1. **Naive** — last observed value
2. **MA(k)** — Moving Average (k = 2, 3, 6)
3. **SES** — Simple Exponential Smoothing (optimised α)
4. **Holt** — Double Exponential Smoothing (trend)
5. **AR(p)** — AutoRegression (p = 1, 2)
6. **ARIMA** — via hand-picked orders
7. **Auto-ARIMA** — pmdarima grid search
8. **SARIMA** — seasonal order (12-month), *use with caution — 19 pts*

> Each model is evaluated on the same 3-fold walk-forward CV used for Prophet.


In [ ]:
# ── Generic walk-forward CV evaluator for statistical models ─────────────
def stat_cv_mape(predict_fn, series, n_splits=3, val_size=2, min_train=10):
    """
    Walk-forward CV for any predict_fn(train_series, n_steps) -> array.
    Returns mean MAPE across folds.
    """
    fold_mapes = []
    for tr_idx, val_idx in ts_cv_splits(series, n_splits=n_splits,
                                         val_size=val_size, min_train=min_train):
        tr  = series.iloc[tr_idx]
        val = series.iloc[val_idx]
        try:
            preds = predict_fn(tr, len(val))
            mape  = safe_mape(val.values, np.array(preds))
            if not np.isnan(mape):
                fold_mapes.append(mape)
        except Exception as e:
            pass  # skip failed folds silently
    return np.mean(fold_mapes) if fold_mapes else np.nan


# ── Forecast on full train data → N_FORECAST steps ───────────────────────
def make_stat_forecast(predict_fn, series, n_forecast):
    """Return a DataFrame with DATE and METRIC_VALUE columns."""
    preds = np.array(predict_fn(series, n_forecast))
    return pd.DataFrame({'DATE': forecast_dates, 'METRIC_VALUE': preds})


In [ ]:
# ==========================================================================
# 9.1  Naive (last value carried forward)
# ==========================================================================
def naive_predict(series, n_steps):
    return [series.iloc[-1]] * n_steps

naive_cv   = stat_cv_mape(naive_predict, train_series)
naive_pred = make_stat_forecast(naive_predict, train_series, N_FORECAST)
results_mape['Naive'] = {'cv_mape': naive_cv, 'test_mape': None}
print(f'Naive          CV MAPE: {naive_cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.2  Moving Average
# ==========================================================================
for k in [2, 3, 6]:
    def _ma(series, n_steps, window=k):
        hist = list(series.values)
        preds = []
        for _ in range(n_steps):
            p = np.mean(hist[-window:])
            preds.append(p)
            hist.append(p)
        return preds
    cv = stat_cv_mape(_ma, train_series)
    results_mape[f'MA({k})'] = {'cv_mape': cv, 'test_mape': None}
    print(f'MA({k})         CV MAPE: {cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.3  Simple Exponential Smoothing  (optimised alpha)
# ==========================================================================
def ses_predict(series, n_steps):
    model = SimpleExpSmoothing(series.values, initialization_method='estimated')
    fit   = model.fit(optimized=True)
    return fit.forecast(n_steps).tolist()

ses_cv   = stat_cv_mape(ses_predict, train_series)
ses_pred = make_stat_forecast(ses_predict, train_series, N_FORECAST)
results_mape['SES'] = {'cv_mape': ses_cv, 'test_mape': None}
print(f'SES            CV MAPE: {ses_cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.4  Holt (Double Exponential Smoothing — additive trend)
# ==========================================================================
def holt_predict(series, n_steps):
    model = ExponentialSmoothing(
        series.values, trend='add', seasonal=None,
        initialization_method='estimated')
    fit = model.fit(optimized=True)
    return fit.forecast(n_steps).tolist()

holt_cv   = stat_cv_mape(holt_predict, train_series)
holt_pred = make_stat_forecast(holt_predict, train_series, N_FORECAST)
results_mape['Holt'] = {'cv_mape': holt_cv, 'test_mape': None}
print(f'Holt           CV MAPE: {holt_cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.5  AutoRegression  AR(p)  p = 1, 2
# ==========================================================================
for p in [1, 2]:
    def _ar(series, n_steps, lags=p):
        model = AutoReg(series.values, lags=lags, old_names=False)
        fit   = model.fit()
        # recursive forecast
        hist  = list(series.values)
        preds = []
        coef  = fit.params            # [const, phi_1, ..., phi_p]
        for _ in range(n_steps):
            pred = coef[0]
            for i in range(1, lags + 1):
                pred += coef[i] * hist[-(i)]
            preds.append(pred)
            hist.append(pred)
        return preds
    cv = stat_cv_mape(_ar, train_series)
    results_mape[f'AR({p})'] = {'cv_mape': cv, 'test_mape': None}
    print(f'AR({p})         CV MAPE: {cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.6  ARIMA  — hand-picked orders most common for monthly macro data
# ==========================================================================
arima_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,0), (0,1,2)]

for order in arima_orders:
    def _arima(series, n_steps, o=order):
        model = ARIMA(series.values, order=o)
        fit   = model.fit()
        return fit.forecast(n_steps).tolist()
    cv  = stat_cv_mape(_arima, train_series)
    name = f'ARIMA{order}'
    results_mape[name] = {'cv_mape': cv, 'test_mape': None}
    print(f'{name:<20} CV MAPE: {cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.7  Auto-ARIMA  (pmdarima — stepwise grid search, no seasonal)
# ==========================================================================
def auto_arima_predict(series, n_steps):
    model = pm.auto_arima(
        series.values,
        seasonal      = False,
        stepwise      = True,
        information_criterion = 'aic',
        error_action  = 'ignore',
        suppress_warnings = True,
        max_p=3, max_q=3, max_d=2
    )
    return model.predict(n_periods=n_steps).tolist()

auto_arima_cv   = stat_cv_mape(auto_arima_predict, train_series)
auto_arima_pred = make_stat_forecast(auto_arima_predict, train_series, N_FORECAST)
results_mape['Auto-ARIMA'] = {'cv_mape': auto_arima_cv, 'test_mape': None}
print(f'Auto-ARIMA     CV MAPE: {auto_arima_cv:.2f}%')


In [ ]:
# ==========================================================================
# 9.8  SARIMA  — seasonal period 12  (caution: only ~19 points)
#         Only run if we have at least 2 full seasonal cycles (24 pts)
# ==========================================================================
if len(train_series) >= 24:
    def sarima_predict(series, n_steps):
        model = SARIMAX(series.values,
                        order=(1,1,0),
                        seasonal_order=(1,0,0,12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        fit = model.fit(disp=False)
        return fit.forecast(n_steps).tolist()

    sarima_cv   = stat_cv_mape(sarima_predict, train_series)
    sarima_pred = make_stat_forecast(sarima_predict, train_series, N_FORECAST)
    results_mape['SARIMA(1,1,0)(1,0,0,12)'] = {
        'cv_mape': sarima_cv, 'test_mape': None}
    print(f'SARIMA         CV MAPE: {sarima_cv:.2f}%')
else:
    print(f'SARIMA skipped — only {len(train_series)} training points '
          '(need ≥ 24 for 12-month seasonal model).')


## 10. Feature Engineering for ML Models

**Conservative feature set (6 features) — designed for ~19 training points:**

| Feature | Type | Rationale |
|---|---|---|
| `lag_1`, `lag_2` | Lag | Direct auto-correlation at t-1, t-2 |
| `rolling_mean_3` | Rolling | Short-term trend smoothing |
| `month_sin`, `month_cos` | Cyclical | Encode calendar month smoothly |
| `trend` | Linear | Global drift index |

> No `lag_3` or longer windows — would eliminate too many training rows.


In [ ]:
# ── Feature builder ───────────────────────────────────────────────────────
def build_features(series):
    """
    Build supervised ML feature matrix from a time series.
    Returns (X, y, feature_names) after dropping NaN rows.
    """
    df_feat = pd.DataFrame({'y': series.values}, index=series.index)
    df_feat.index = pd.to_datetime(df_feat.index)

    df_feat['lag_1']         = df_feat['y'].shift(1)
    df_feat['lag_2']         = df_feat['y'].shift(2)
    df_feat['rolling_mean_3']= df_feat['y'].shift(1).rolling(3).mean()
    df_feat['month_sin']     = np.sin(2 * np.pi * df_feat.index.month / 12)
    df_feat['month_cos']     = np.cos(2 * np.pi * df_feat.index.month / 12)
    df_feat['trend']         = np.arange(len(df_feat))

    df_feat = df_feat.dropna()
    feature_names = ['lag_1', 'lag_2', 'rolling_mean_3',
                     'month_sin', 'month_cos', 'trend']
    X = df_feat[feature_names].values
    y = df_feat['y'].values
    return X, y, feature_names


# ── Recursive multi-step forecaster ───────────────────────────────────────
def recursive_ml_forecast(model, train_series, forecast_dates,
                           scaler=None, scale_target=False,
                           y_scaler=None):
    """
    Predict n steps ahead recursively:
      at each step, append the prediction to history and recompute features.
    """
    history      = list(train_series.values)
    history_idx  = list(pd.to_datetime(train_series.index))
    predictions  = []

    for fd in pd.to_datetime(forecast_dates):
        lag_1          = history[-1]
        lag_2          = history[-2] if len(history) >= 2 else history[-1]
        rolling_mean_3 = np.mean(history[-3:]) if len(history) >= 3 else np.mean(history)
        month_sin      = np.sin(2 * np.pi * fd.month / 12)
        month_cos      = np.cos(2 * np.pi * fd.month / 12)
        trend          = len(history)

        feat = np.array([[lag_1, lag_2, rolling_mean_3,
                          month_sin, month_cos, trend]])

        if scaler is not None:
            feat = scaler.transform(feat)

        pred = model.predict(feat)[0]

        if y_scaler is not None:
            pred = y_scaler.inverse_transform([[pred]])[0][0]

        pred = max(pred, 0)  # non-negative constraint
        predictions.append(pred)
        history.append(pred)
        history_idx.append(fd)

    return predictions


# ── ML walk-forward CV ────────────────────────────────────────────────────
def ml_cv_mape(model_factory, series, n_splits=3, val_size=2, min_train=10,
               use_scaler=False):
    """
    Walk-forward CV for ML model.
    model_factory() returns a fresh (unfitted) model each fold.
    """
    fold_mapes = []
    for tr_idx, val_idx in ts_cv_splits(series, n_splits=n_splits,
                                         val_size=val_size, min_train=min_train):
        tr_s  = series.iloc[tr_idx]
        val_s = series.iloc[val_idx]

        X_tr, y_tr, fnames = build_features(tr_s)
        if len(X_tr) < 4:
            continue

        scaler = StandardScaler() if use_scaler else None
        if scaler:
            X_tr = scaler.fit_transform(X_tr)

        try:
            mdl = model_factory()
            mdl.fit(X_tr, y_tr)
            val_dates = pd.date_range(
                start=pd.Timestamp(tr_s.index.max()) + pd.DateOffset(months=1),
                periods=len(val_s), freq='MS').date
            preds = recursive_ml_forecast(
                mdl, tr_s, val_dates, scaler=scaler)
            mape  = safe_mape(val_s.values, np.array(preds))
            if not np.isnan(mape):
                fold_mapes.append(mape)
        except Exception:
            pass

    return np.mean(fold_mapes) if fold_mapes else np.nan


# Build final feature matrix
X_train_ml, y_train_ml, feat_names = build_features(train_series)
print(f'ML feature matrix shape : {X_train_ml.shape}')
print(f'Features                : {feat_names}')


## 11. ML Models

All models use **standardised features** (zero-mean, unit-variance).  
Regularisation parameters are set conservatively given the small training set.

| Model | Key settings | Why suitable for small n |
|---|---|---|
| Ridge | alpha tuned via LOO-CV | L2 regularisation prevents blow-up |
| Lasso | alpha tuned via LOO-CV | L1 sparsity — feature selection |
| ElasticNet | l1_ratio=0.5 | balanced L1+L2 |
| SVR (RBF) | C=1, epsilon=0.1 | good with normalised inputs |
| XGBoost | max_depth=2, n_estimators=50 | shallow trees → low variance |
| LightGBM | num_leaves=4, n_estimators=50 | similar rationale |


In [ ]:
# ── Fit shared scaler on full training features ───────────────────────────
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train_ml)


In [ ]:
# ==========================================================================
# 11.1  Ridge Regression
# ==========================================================================
from sklearn.linear_model import RidgeCV

ridge_alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 500.0]
ridge_model  = RidgeCV(alphas=ridge_alphas, cv=3)
ridge_model.fit(X_train_scaled, y_train_ml)
print(f'Ridge best alpha : {ridge_model.alpha_}')

ridge_cv    = ml_cv_mape(lambda: Ridge(alpha=ridge_model.alpha_),
                         train_series, use_scaler=True)
ridge_preds = recursive_ml_forecast(
    ridge_model, train_series, forecast_dates, scaler=scaler_ml)

results_mape['Ridge'] = {'cv_mape': ridge_cv, 'test_mape': None}
print(f'Ridge          CV MAPE: {ridge_cv:.2f}%')


In [ ]:
# ==========================================================================
# 11.2  Lasso Regression
# ==========================================================================
from sklearn.linear_model import LassoCV

lasso_model = LassoCV(cv=3, max_iter=5000)
lasso_model.fit(X_train_scaled, y_train_ml)
print(f'Lasso best alpha : {lasso_model.alpha_:.6f}')

lasso_cv    = ml_cv_mape(lambda: Lasso(alpha=lasso_model.alpha_, max_iter=5000),
                         train_series, use_scaler=True)
lasso_preds = recursive_ml_forecast(
    lasso_model, train_series, forecast_dates, scaler=scaler_ml)

results_mape['Lasso'] = {'cv_mape': lasso_cv, 'test_mape': None}
print(f'Lasso          CV MAPE: {lasso_cv:.2f}%')


In [ ]:
# ==========================================================================
# 11.3  ElasticNet
# ==========================================================================
from sklearn.linear_model import ElasticNetCV

enet_model = ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=3, max_iter=5000)
enet_model.fit(X_train_scaled, y_train_ml)
print(f'ElasticNet best alpha={enet_model.alpha_:.6f}, l1_ratio={enet_model.l1_ratio_}')

enet_cv    = ml_cv_mape(
    lambda: ElasticNet(alpha=enet_model.alpha_,
                       l1_ratio=enet_model.l1_ratio_, max_iter=5000),
    train_series, use_scaler=True)
enet_preds = recursive_ml_forecast(
    enet_model, train_series, forecast_dates, scaler=scaler_ml)

results_mape['ElasticNet'] = {'cv_mape': enet_cv, 'test_mape': None}
print(f'ElasticNet     CV MAPE: {enet_cv:.2f}%')


In [ ]:
# ==========================================================================
# 11.4  Support Vector Regression (RBF kernel)
# ==========================================================================
from sklearn.svm import SVR

svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.05)
svr_model.fit(X_train_scaled, y_train_ml)

svr_cv    = ml_cv_mape(lambda: SVR(kernel='rbf', C=1.0, epsilon=0.05),
                       train_series, use_scaler=True)
svr_preds = recursive_ml_forecast(
    svr_model, train_series, forecast_dates, scaler=scaler_ml)

results_mape['SVR_RBF'] = {'cv_mape': svr_cv, 'test_mape': None}
print(f'SVR (RBF)      CV MAPE: {svr_cv:.2f}%')


In [ ]:
# ==========================================================================
# 11.5  XGBoost  (shallow trees — essential for small n)
# ==========================================================================
xgb_model = XGBRegressor(
    n_estimators    = 50,
    max_depth       = 2,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    reg_alpha       = 1.0,    # L1
    reg_lambda      = 5.0,    # L2  — heavy regularisation
    random_state    = 42,
    verbosity       = 0
)
xgb_model.fit(X_train_scaled, y_train_ml)

xgb_cv    = ml_cv_mape(
    lambda: XGBRegressor(n_estimators=50, max_depth=2, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=1.0, reg_lambda=5.0,
                         random_state=42, verbosity=0),
    train_series, use_scaler=True)
xgb_preds = recursive_ml_forecast(
    xgb_model, train_series, forecast_dates, scaler=scaler_ml)

results_mape['XGBoost'] = {'cv_mape': xgb_cv, 'test_mape': None}
print(f'XGBoost        CV MAPE: {xgb_cv:.2f}%')


In [ ]:
# ==========================================================================
# 11.6  LightGBM  (gradient boosting — small memory, fast)
# ==========================================================================
lgb_model = lgb.LGBMRegressor(
    n_estimators    = 50,
    num_leaves      = 4,      # very shallow
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    reg_alpha       = 1.0,
    reg_lambda      = 5.0,
    random_state    = 42,
    verbose         = -1
)
lgb_model.fit(X_train_scaled, y_train_ml)

lgb_cv    = ml_cv_mape(
    lambda: lgb.LGBMRegressor(n_estimators=50, num_leaves=4,
                              learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, reg_alpha=1.0,
                              reg_lambda=5.0, random_state=42, verbose=-1),
    train_series, use_scaler=True)
lgb_preds = recursive_ml_forecast(
    lgb_model, train_series, forecast_dates, scaler=scaler_ml)

results_mape['LightGBM'] = {'cv_mape': lgb_cv, 'test_mape': None}
print(f'LightGBM       CV MAPE: {lgb_cv:.2f}%')


In [ ]:
# ── XGBoost feature importances ───────────────────────────────────────────
fi = pd.Series(xgb_model.feature_importances_, index=feat_names).sort_values()
fi.plot(kind='barh', figsize=(8, 4), title='XGBoost Feature Importances')
plt.tight_layout(); plt.show()

fi_lgb = pd.Series(lgb_model.feature_importances_, index=feat_names).sort_values()
fi_lgb.plot(kind='barh', figsize=(8, 4), title='LightGBM Feature Importances')
plt.tight_layout(); plt.show()


## 12. Results Comparison

In [ ]:
# ==========================================================================
# 12.1  Summary MAPE table
# ==========================================================================
benchmark_mapes = {
    'Statistical_2025_0+12': 3.21,
    'Senior_Prophet':        6.54,
    'Original_Forecast':     395.19,
}

rows = []
for name, vals in results_mape.items():
    rows.append({
        'Model'      : name,
        'CV MAPE (%)': round(vals['cv_mape'], 2) if vals['cv_mape'] else np.nan,
        'Test MAPE (%)': round(vals['test_mape'], 2)
                        if vals.get('test_mape') is not None else '—',
        'Type'       : 'ML' if name in [
            'Ridge','Lasso','ElasticNet','SVR_RBF','XGBoost','LightGBM']
                       else 'Prophet' if 'Prophet' in name
                       else 'Statistical'
    })

results_df = pd.DataFrame(rows).sort_values('CV MAPE (%)')

print('\n' + '='*65)
print(f"{'Model':<30} {'CV MAPE':>10}  {'Test MAPE':>10}  {'Type':<14}")
print('='*65)
for _, row in results_df.iterrows():
    flag = ' ★' if isinstance(row['CV MAPE (%)'], float) and row['CV MAPE (%)'] < 3.21 else ''
    print(f"{row['Model']:<30} {row['CV MAPE (%)']:>10}  "
          f"{str(row['Test MAPE (%)']):>10}  {row['Type']:<14}{flag}")
print('-'*65)
print(f"{'Statistical_2025_0+12 [TARGET]':<30} {'3.21':>10}  {'—':>10}  {'Benchmark':<14}")
print(f"{'Senior_Prophet [BASELINE]':<30} {'6.54':>10}  {'—':>10}  {'Benchmark':<14}")
print('='*65)
print('★ = beats the 3.21% target')

results_df


In [ ]:
# ==========================================================================
# 12.2  Horizontal bar chart of CV MAPEs
# ==========================================================================
plot_df = results_df[results_df['CV MAPE (%)'].apply(
    lambda x: isinstance(x, (int, float)) and not np.isnan(x))].copy()

colors = {
    'Prophet'    : '#4C72B0',
    'Statistical': '#55A868',
    'ML'         : '#DD8452'
}
bar_colors = [colors.get(t, 'grey') for t in plot_df['Type']]

fig, ax = plt.subplots(figsize=(12, max(6, len(plot_df) * 0.5)))
ax.barh(plot_df['Model'], plot_df['CV MAPE (%)'], color=bar_colors, edgecolor='white')
ax.axvline(3.21,  color='green',  linestyle='--', linewidth=1.5, label='Target 3.21%')
ax.axvline(6.54,  color='orange', linestyle='--', linewidth=1.5, label='Senior Prophet 6.54%')
ax.set_xlabel('CV MAPE (%)')
ax.set_title(f'{target_metric} — Model CV MAPE Comparison ({portfolio})')
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4C72B0', label='Prophet'),
    Patch(facecolor='#55A868', label='Statistical'),
    Patch(facecolor='#DD8452', label='ML'),
]
ax.legend(handles=legend_elements + [
    plt.Line2D([0],[0],color='green',  linestyle='--', label='Target 3.21%'),
    plt.Line2D([0],[0],color='orange', linestyle='--', label='Senior Prophet 6.54%'),
])
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================================
# 12.3  Forecast trajectories for top-5 models vs training data
# ==========================================================================
top5 = results_df.head(5)['Model'].tolist()

# Map model name → prediction array
model_predictions = {
    'Prophet_Tuned' : prophet_tuned_pred['METRIC_VALUE'].values,
    'SES'           : ses_pred['METRIC_VALUE'].values,
    'Holt'          : holt_pred['METRIC_VALUE'].values,
    'Auto-ARIMA'    : auto_arima_pred['METRIC_VALUE'].values,
    'Ridge'         : np.array(ridge_preds),
    'Lasso'         : np.array(lasso_preds),
    'ElasticNet'    : np.array(enet_preds),
    'SVR_RBF'       : np.array(svr_preds),
    'XGBoost'       : np.array(xgb_preds),
    'LightGBM'      : np.array(lgb_preds),
}

fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(pd.to_datetime(train_series.index), train_series.values,
        'ko-', label='Actual (train)', linewidth=2, markersize=5)
if len(test_series) > 0:
    ax.plot(pd.to_datetime(test_series.index), test_series.values,
            'k^--', label='Actual (test)', linewidth=2, markersize=7)

cmap = plt.cm.tab10
for i, mname in enumerate(top5):
    if mname in model_predictions:
        ax.plot(pd.to_datetime(forecast_dates), model_predictions[mname],
                marker='o', linestyle='--', color=cmap(i),
                label=f'{mname} ({results_mape[mname]["cv_mape"]:.2f}%)',
                alpha=0.85)

ax.set_title(f'{target_metric} — Top 5 Model Forecasts ({portfolio})')
ax.set_xlabel('Date')
ax.set_ylabel('Value')
ax.legend(loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================================================
# 12.4  Best model — full prediction output in existing notebook format
# ==========================================================================
best_model_name = results_df.iloc[0]['Model']
best_model_cv   = results_df.iloc[0]['CV MAPE (%)']
print(f'Best model : {best_model_name}  (CV MAPE = {best_model_cv:.2f}%)')
print(f'Target     : 3.21%')
print(f'Improvement over senior Prophet: {6.54 - best_model_cv:.2f}pp')

# Build final prediction in existing notebook format
if best_model_name in model_predictions:
    best_preds_arr = model_predictions[best_model_name]
elif best_model_name == 'Prophet_Tuned':
    best_preds_arr = prophet_tuned_pred['METRIC_VALUE'].values

# Construct dataframe matching add_reforecast_data format
best_pred_df = pd.DataFrame({
    'DATE'         : list(train_series.index) + list(forecast_dates),
    'METRIC_VALUE' : list(train_series.values) + list(best_preds_arr),
    'FORECAST_TYPE': ['Actual'] * len(train_series) + [best_model_name] * N_FORECAST
})

plot_results(best_pred_df, target_metric)
